# Senologie Diagnose Analysis

Query Aidbox FHIR Server for breast cancer diagnoses.

**Endpoints (inside Docker):**
- Aidbox: `http://aidbox:8888`
- PostgreSQL: `aidbox-db:5432`

In [ ]:
import os
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth
from sqlalchemy import create_engine

# Auto-detect environment (Docker vs local)
AIDBOX_URL = os.environ.get('AIDBOX_URL', 'http://localhost:8888')
PGHOST = os.environ.get('PGHOST', 'localhost')
PGPORT = os.environ.get('PGPORT', '5437')  # 5437 for local, 5432 inside Docker

AUTH = HTTPBasicAuth('root', 'secret')
PG_CONN = f"postgresql://postgres:postgres@{PGHOST}:{PGPORT}/aidbox"

print(f"Aidbox: {AIDBOX_URL}")
print(f"PostgreSQL: {PGHOST}:{PGPORT}")

engine = create_engine(PG_CONN)

## 1. FHIR REST API

In [ ]:
response = requests.get(f"{AIDBOX_URL}/fhir/Condition", auth=AUTH)
bundle = response.json()

print(f"Found {bundle.get('total', len(bundle.get('entry', [])))} conditions")

records = []
for entry in bundle.get('entry', []):
    r = entry['resource']
    snomed = next((c for c in r.get('code', {}).get('coding', []) 
                   if c.get('system') == 'http://snomed.info/sct'), {})
    records.append({
        'id': r.get('id'),
        'snomed_code': snomed.get('code'),
        'diagnosis': snomed.get('display'),
        'status': r.get('clinicalStatus', {}).get('coding', [{}])[0].get('code'),
    })

pd.DataFrame(records)

## 2. Direct PostgreSQL Query

Using the ViewDefinition schema from `ViewDefinition-senologie-diagnose.json`

In [ ]:
query = """
SELECT 
    c.id as condition_id,
    c.resource#>>'{subject,reference}' as patient_id,
    (SELECT cd->>'code' FROM jsonb_array_elements(c.resource->'code'->'coding') cd 
     WHERE cd->>'system' = 'http://snomed.info/sct' LIMIT 1) as snomed_code,
    (SELECT cd->>'display' FROM jsonb_array_elements(c.resource->'code'->'coding') cd 
     WHERE cd->>'system' = 'http://snomed.info/sct' LIMIT 1) as diagnosis,
    (SELECT cd->>'code' FROM jsonb_array_elements(c.resource->'code'->'coding') cd 
     WHERE cd->>'system' = 'http://fhir.de/CodeSystem/bfarm/icd-10-gm' LIMIT 1) as icd10_code,
    c.resource#>>'{bodySite,0,coding,0,display}' as laterality,
    c.resource#>>'{clinicalStatus,coding,0,code}' as clinical_status,
    c.resource#>>'{verificationStatus,coding,0,code}' as verification_status,
    c.resource->>'recordedDate' as recorded_date
FROM condition c
"""

df = pd.read_sql(query, engine)
df

## 3. Analysis

In [ ]:
print("Diagnosis Distribution:")
print(df['diagnosis'].value_counts())

print("\nClinical Status:")
print(df['clinical_status'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['diagnosis'].value_counts().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Diagnoses')
axes[0].set_xlabel('Count')

df['clinical_status'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.0f%%')
axes[1].set_title('Clinical Status')

plt.tight_layout()
plt.show()